# vLLM paper-classification performance test

This notebook benchmarks concurrent `PaperClassifier` requests against a small read-only sample of pending papers. It defaults to 10 papers, records per-paper latency and failures, and does **not** update SQLite.

Start the OpenAI-compatible vLLM server, then run the cells from top to bottom.

In [ ]:
from __future__ import annotations

import asyncio
import os
import socket
import sqlite3
import statistics
import sys
import time
from pathlib import Path

from IPython.display import display

PROJECT_ROOT = Path("/nobackup/proj/disk/naiss2024-5-630/personal/george/synth_extract")
DB_PATH = PROJECT_ROOT / "data" / "central_workspace.db"
LIMIT = 10
MODEL = os.getenv("LLM_MODEL", "qwen3.6-27b")
BASE_URL = os.getenv("LLM_BASE_URL", "http://127.0.0.1:8000/v1")
API_KEY = os.getenv("LLM_API_KEY", "not-required")
TIMEOUT = 60.0
MAX_TOKENS = 10_000
MAX_PARALLEL_REQUESTS = int(os.getenv("MAX_PARALLEL_REQUESTS", "8"))

if LIMIT <= 0:
    raise ValueError("LIMIT must be greater than zero")
if MAX_PARALLEL_REQUESTS <= 0:
    raise ValueError("MAX_PARALLEL_REQUESTS must be greater than zero")
if not DB_PATH.is_file():
    raise FileNotFoundError(DB_PATH)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

display({
    "host": socket.gethostname(),
    "python": sys.executable,
    "slurm_job": os.environ.get("SLURM_JOB_ID"),
    "database": str(DB_PATH),
    "limit": LIMIT,
    "model": MODEL,
    "base_url": BASE_URL,
    "timeout_seconds": TIMEOUT,
    "max_tokens": MAX_TOKENS,
    "max_parallel_requests": MAX_PARALLEL_REQUESTS,
})

In [ ]:
from synth_extract.agents.classification import (
    ClassificationFailure,
    ClassificationResult,
    PaperClassifier,
)

classifier = PaperClassifier(
    model=MODEL,
    base_url=BASE_URL,
    api_key=API_KEY,
    temperature=0.0,
    timeout=TIMEOUT,
    max_tokens=MAX_TOKENS,
)

display(classifier.llm_config())
health = classifier.health_check()
if isinstance(health, ClassificationFailure):
    raise RuntimeError(
        f"vLLM health check failed ({health.error_type}): {health.message}"
    )
print("vLLM health check passed")

In [ ]:
TABLE = "papers"
QWEN_PREFIX = "qwen"


def quote_identifier(identifier: str) -> str:
    return '"' + identifier.replace('"', '""') + '"'


def find_qwen_column(connection: sqlite3.Connection) -> str:
    columns = connection.execute(
        f"PRAGMA table_info({quote_identifier(TABLE)})"
    ).fetchall()
    names = [str(column[1]) for column in columns]
    required = {"paper_id", "paper_uid", "title", "abstract"}
    missing = sorted(required.difference(names))
    if missing:
        raise RuntimeError(f"Missing required columns: {', '.join(missing)}")

    matches = [name for name in names if name.lower().startswith(QWEN_PREFIX)]
    if len(matches) != 1:
        raise RuntimeError(
            f"Expected one column beginning with {QWEN_PREFIX!r}; found {matches}"
        )
    return matches[0]


def normalize(value: object) -> str:
    return "" if value is None else str(value)


# Read-only mode prevents this benchmark from changing classification labels.
database_uri = f"file:{DB_PATH}?mode=ro"
with sqlite3.connect(database_uri, uri=True) as connection:
    result_column = find_qwen_column(connection)
    table_sql = quote_identifier(TABLE)
    column_sql = quote_identifier(result_column)
    pending = int(
        connection.execute(
            f"SELECT COUNT(*) FROM {table_sql} WHERE {column_sql} IS NULL"
        ).fetchone()[0]
    )
    rows = connection.execute(
        f"""
        SELECT paper_id, paper_uid, title, abstract
        FROM {table_sql}
        WHERE {column_sql} IS NULL
        ORDER BY paper_id
        LIMIT ?
        """,
        (LIMIT,),
    ).fetchall()

papers = [
    {
        "paper_id": int(paper_id),
        "paper_uid": normalize(paper_uid),
        "title": normalize(title),
        "abstract": normalize(abstract),
    }
    for paper_id, paper_uid, title, abstract in rows
]

print(f"Result column: {result_column}")
print(f"Pending papers: {pending:,}")
print(f"Selected papers: {len(papers)}")
display([
    {
        "paper_id": paper["paper_id"],
        "paper_uid": paper["paper_uid"],
        "title": paper["title"],
        "abstract_characters": len(paper["abstract"]),
    }
    for paper in papers
])

## Synchronous benchmark

Classify the selected papers one at a time using `classifier.classify()`.

In [ ]:
if not papers:
    raise RuntimeError("No pending papers were found to benchmark")

sync_results = []
sync_benchmark_started = time.perf_counter()

for index, paper in enumerate(papers, start=1):
    request_started = time.perf_counter()
    try:
        outcome = classifier.classify(
            title=str(paper["title"]),
            abstract=str(paper["abstract"]),
        )
    except Exception as exc:
        outcome = None
        unexpected_error = f"{type(exc).__name__}: {exc}"
    else:
        unexpected_error = None
    latency_seconds = time.perf_counter() - request_started

    record = {
        "input_index": index - 1,
        "paper_id": paper["paper_id"],
        "paper_uid": paper["paper_uid"],
        "title": paper["title"],
        "queue_seconds": 0.0,
        "latency_seconds": latency_seconds,
        "success": isinstance(outcome, ClassificationResult),
        "label": outcome.label if isinstance(outcome, ClassificationResult) else None,
        "error_type": (
            outcome.error_type
            if isinstance(outcome, ClassificationFailure)
            else "unexpected" if unexpected_error else None
        ),
        "error_message": (
            outcome.message
            if isinstance(outcome, ClassificationFailure)
            else unexpected_error
        ),
    }
    sync_results.append(record)

    status = (
        f"label={record['label']}"
        if record["success"]
        else f"error={record['error_type']}"
    )
    print(
        f"[{index:>2}/{len(papers)}] paper_id={paper['paper_id']} "
        f"{status} latency={latency_seconds:.3f}s"
    )

sync_wall_seconds = time.perf_counter() - sync_benchmark_started
print(f"Synchronous wall time: {sync_wall_seconds:.3f}s")
print(
    f"Synchronous average wall time per paper: "
    f"{sync_wall_seconds / len(sync_results):.3f}s"
)

## Asynchronous benchmark

Classify the same papers concurrently using `classifier.aclassify()`, bounded by `MAX_PARALLEL_REQUESTS`.

In [ ]:
if not papers:
    raise RuntimeError("No pending papers were found to benchmark")

# This client-side semaphore independently limits in-flight HTTP requests.
request_semaphore = asyncio.Semaphore(MAX_PARALLEL_REQUESTS)


async def classify_one(index: int, paper: dict[str, object]) -> dict[str, object]:
    queued_at = time.perf_counter()
    async with request_semaphore:
        queue_seconds = time.perf_counter() - queued_at
        request_started = time.perf_counter()
        try:
            outcome = await classifier.aclassify(
                title=str(paper["title"]),
                abstract=str(paper["abstract"]),
            )
        except Exception as exc:
            outcome = None
            unexpected_error = f"{type(exc).__name__}: {exc}"
        else:
            unexpected_error = None
        latency_seconds = time.perf_counter() - request_started

    return {
        "input_index": index,
        "paper_id": paper["paper_id"],
        "paper_uid": paper["paper_uid"],
        "title": paper["title"],
        "queue_seconds": queue_seconds,
        "latency_seconds": latency_seconds,
        "success": isinstance(outcome, ClassificationResult),
        "label": outcome.label if isinstance(outcome, ClassificationResult) else None,
        "error_type": (
            outcome.error_type
            if isinstance(outcome, ClassificationFailure)
            else "unexpected" if unexpected_error else None
        ),
        "error_message": (
            outcome.message
            if isinstance(outcome, ClassificationFailure)
            else unexpected_error
        ),
    }


async_benchmark_started = time.perf_counter()
tasks = [
    asyncio.create_task(classify_one(index, paper))
    for index, paper in enumerate(papers)
]

async_results = []
for completed_count, task in enumerate(asyncio.as_completed(tasks), start=1):
    record = await task
    async_results.append(record)
    status = (
        f"label={record['label']}"
        if record["success"]
        else f"error={record['error_type']}"
    )
    print(
        f"[{completed_count:>2}/{len(papers)}] "
        f"paper_id={record['paper_id']} {status} "
        f"latency={record['latency_seconds']:.3f}s "
        f"queued={record['queue_seconds']:.3f}s"
    )

async_wall_seconds = time.perf_counter() - async_benchmark_started
async_results.sort(key=lambda record: record["input_index"])
print(f"Asynchronous wall time: {async_wall_seconds:.3f}s")
print(
    f"Asynchronous average wall time per paper: "
    f"{async_wall_seconds / len(async_results):.3f}s"
)

## Synchronous versus asynchronous comparison

In [ ]:
def percentile(values: list[float], percentile_value: float) -> float:
    ordered = sorted(values)
    if len(ordered) == 1:
        return ordered[0]
    position = (len(ordered) - 1) * percentile_value / 100
    lower = int(position)
    upper = min(lower + 1, len(ordered) - 1)
    fraction = position - lower
    return ordered[lower] + (ordered[upper] - ordered[lower]) * fraction


def summarize_benchmark(
    mode: str,
    records: list[dict[str, object]],
    wall_seconds: float,
    max_parallel_requests: int,
) -> dict[str, object]:
    latencies = [float(record["latency_seconds"]) for record in records]
    successful = [record for record in records if record["success"]]
    return {
        "mode": mode,
        "papers_attempted": len(records),
        "max_parallel_requests": max_parallel_requests,
        "successful": len(successful),
        "failed": len(records) - len(successful),
        "success_rate_percent": round(100 * len(successful) / len(records), 2),
        "true_labels": sum(record["label"] is True for record in successful),
        "false_labels": sum(record["label"] is False for record in successful),
        "total_wall_seconds": round(wall_seconds, 3),
        "average_wall_seconds_per_paper": round(wall_seconds / len(records), 3),
        "throughput_papers_per_second": round(len(records) / wall_seconds, 3),
        "mean_request_latency_seconds": round(statistics.mean(latencies), 3),
        "median_request_latency_seconds": round(statistics.median(latencies), 3),
        "p95_request_latency_seconds": round(percentile(latencies, 95), 3),
        "min_request_latency_seconds": round(min(latencies), 3),
        "max_request_latency_seconds": round(max(latencies), 3),
    }


sync_summary = summarize_benchmark(
    mode="synchronous",
    records=sync_results,
    wall_seconds=sync_wall_seconds,
    max_parallel_requests=1,
)
async_summary = summarize_benchmark(
    mode="asynchronous",
    records=async_results,
    wall_seconds=async_wall_seconds,
    max_parallel_requests=min(MAX_PARALLEL_REQUESTS, len(async_results)),
)
comparison = [sync_summary, async_summary]
speedup = sync_wall_seconds / async_wall_seconds

print(f"Asynchronous wall-time speedup: {speedup:.2f}x")
try:
    import pandas as pd
except ImportError:
    display(comparison)
else:
    display(pd.DataFrame(comparison).set_index("mode"))

result_rows = [
    {
        "mode": mode,
        **record,
        "queue_seconds": round(float(record["queue_seconds"]), 3),
        "latency_seconds": round(float(record["latency_seconds"]), 3),
    }
    for mode, records in (
        ("synchronous", sync_results),
        ("asynchronous", async_results),
    )
    for record in records
]
if 'pd' in globals():
    display(pd.DataFrame(result_rows))
else:
    display(result_rows)

failures = [record for record in result_rows if not record["success"]]
if failures:
    print("Failures:")
    display(failures)

## Interpreting the result

- The synchronous section uses `PaperClassifier.classify()` one paper at a time.
- The asynchronous section uses `PaperClassifier.aclassify()` with at most `MAX_PARALLEL_REQUESTS` in-flight HTTP requests.
- `MAX_PARALLEL_REQUESTS` is a client-side limit and may be lower than the vLLM server's `--max-num-seqs 64` setting.
- `latency_seconds` measures active request time; `queue_seconds` measures time waiting for a client concurrency slot.
- `average_wall_seconds_per_paper` is total wall time for the selected `LIMIT` divided by the number of attempted papers.
- `mean_request_latency_seconds` is the average latency observed by individual requests; under concurrency it is not the same as wall time per paper.
- The first completion may include model or cache warm-up overhead.
- Increase `LIMIT` and `MAX_PARALLEL_REQUESTS` gradually. The database remains read-only.
- When finished with the classifier, close its clients with `classifier.close()` and `await classifier.aclose()`.